# 第 2 周第 2 天实验 —— 用 Ollama + Gradio 生成公司宣传册

## 练习目标

把**网页抓取 + 流式 Chat Completions + Gradio UI**串起来：输入公司名与官网 URL，选本地模型（`llama` / `phi`），流式生成一份面向客户 / 投资人 / 求职者的 Markdown 宣传册。

## 和本课 Week 2 Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Ollama OpenAI 兼容接口 | `base_url=http://localhost:11434/v1` |
| 流式输出 `stream=True` | 累加 `delta.content` 并 `yield` |
| 多模型切换 | `stream_llama` / `stream_phi` + Dropdown |
| Gradio Interface | 输入框 + 示例 + `launch()` |

## 怎么跑

1. 确保本机 Ollama 已启动，并已拉取 `llama3.2:1b`、`phi`
2. 同目录需有 `scraper.fetch_website_contents`（抓取落地页正文）
3. 从上到下运行单元格，在 Gradio 里填公司名与 URL，或点 examples


In [ ]:
# ========== 导入与客户端：接本地 Ollama，准备宣传册 system 提示 ==========

# 从 openai 导入 OpenAI 客户端：通过 OpenAI 兼容协议调用本地 Ollama
from openai import OpenAI
# 导入 gradio：快速搭 Web UI（表单 + 流式 Markdown 输出）
import gradio as gr
# 创建客户端：base_url 指向本机 Ollama 的 /v1；api_key 仅占位（Ollama 不校验，但 SDK 要求有值）
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# 从同目录 scraper 导入抓取函数：把落地页 HTML 抽成文本，供后续塞进 user prompt
from scraper import fetch_website_contents

# system_message：定角色与输出格式（英文 prompt 必须原样保留，影响模型行为）
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""


In [ ]:
# ========== 路径 A：用本地 llama3.2:1b 流式生成宣传册 ==========

# stream_llama：接收完整 user prompt，以生成器形式逐块产出「迄今累计」的 Markdown
def stream_llama(prompt):
    # messages：system 定「怎么写册子」，user 放公司名 + 落地页正文
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    # Chat Completions：stream=True → 服务端持续推送 token/片段
    stream = client.chat.completions.create(
        model='llama3.2:1b',
        messages=messages,
        stream=True
    )
    # result：累积已生成文本；Gradio 流式输出需要「完整前缀」而不是单片 delta
    result = ""
    # 遍历每个 chunk；delta.content 可能为 None，用 or "" 兜底
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        # yield 累计字符串：界面上呈现「打字机」不断变长的效果
        yield result


In [ ]:
# ========== 路径 B：用本地 phi 流式生成（结构与 llama 相同，只换模型名） ==========

# stream_phi：与 stream_llama 对称，便于 Dropdown 切换对比风格/速度
def stream_phi(prompt):
    # 同样组装 system + user；prompt 内容由上层 stream_brochure 拼好
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    # 模型改为 phi；其余参数（stream=True）保持一致
    stream = client.chat.completions.create(
        model='phi',
        messages=messages,
        stream=True
    )
    # 同样累加并 yield，保证 Gradio 接口签名一致
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [ ]:
# ========== 编排：抓网页 → 选模型 → 把流式生成器交给 Gradio ==========

# stream_brochure：Gradio Interface 的 fn；参数顺序须与 inputs 一致
def stream_brochure(company_name, url, model):
    # 先 yield 空串：立刻清空/刷新输出区，避免上一轮残留
    yield ""
    # 拼 user prompt：英文模板保持原样；后面接抓取到的落地页正文
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    # 按 Dropdown 的值分发到对应流式函数（字符串比较逻辑不改）
    if model=="llama":
        result = stream_llama(prompt)
    elif model=="phi":
        result = stream_phi(prompt)
    else:
        # 未知模型名：显式报错（错误文案保持英文原样）
        raise ValueError("Unknown model")
    # yield from：把子生成器的每个累计片段继续往外抛给 Gradio
    yield from result


In [ ]:
# ========== Gradio UI：公司名 / URL / 模型选择 → 流式 Markdown ==========

# 公司名输入框（label 文案影响 UI，保持原样）
name_input = gr.Textbox(label="Company name:")
# 落地页 URL；需含 http:// 或 https://（提示写在 label 里）
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
# 模型下拉：值 "llama" / "phi" 对应上面 if/elif 分支；默认 llama
model_selector = gr.Dropdown(["llama", "phi"], label="Select model", value="llama")
# 输出区用 Markdown，便于渲染宣传册标题/列表
message_output = gr.Markdown(label="Response:")

# Interface：把 stream_brochure 绑到三输入一输出；examples 可一键填表
view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "llama"],
            ["Edward Donner", "https://edwarddonner.com", "phi"]
        ],
    # 关闭 flagging（报告不良输出）按钮，界面更干净
    flagging_mode="never"
    )
# 启动本地 Web 服务；笔记本里会打印访问链接
view.launch()
